In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import random, pickle, math

device = (
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f"Device: {device}")

# Hyperparameters
batch_size    = 32
block_size    = 128    # BPE token dày hơn char
max_iters     = 500
learning_rate = 3e-4
eval_iters    = 50
n_embd        = 128
n_head        = 4
n_layer       = 3
dropout       = 0.1

Device: cpu


**Chọn backend:**
- `backend = 'custom'` — BPE thuần Python từ `bpe_tokenizer.py`
- `backend = 'hf'`     — HuggingFace `tokenizers` (`pip install tokenizers`)

In [ ]:
from bpe_tokenizer import BPETokenizer, HFTokenizerWrapper

# Chọn backend 
backend    = 'custom'        # 'custom' | 'hf'
VOCAB_SIZE = 1000            # kích thước vocab BPE
DATA_FILE  = 'wizard_of_oz.txt'

if backend == 'custom':
    import os
    if os.path.exists('bpe_vocab.pkl'):
        tokenizer = BPETokenizer.load('bpe_vocab.pkl')
    else:
        with open(DATA_FILE, 'r', encoding='utf-8') as f:
            raw = f.read()
        tokenizer = BPETokenizer()
        tokenizer.train(raw, vocab_size=VOCAB_SIZE)
        tokenizer.save('bpe_vocab.pkl')

elif backend == 'hf':
    import os
    if os.path.exists('hf_bpe/tokenizer.json'):
        tokenizer = HFTokenizerWrapper.load('hf_bpe')
    else:
        tokenizer = HFTokenizerWrapper()
        tokenizer.train(DATA_FILE, vocab_size=VOCAB_SIZE)
        tokenizer.save('hf_bpe')

# Interface đồng nhất với code cũ
encode     = tokenizer.encode
decode     = tokenizer.decode
vocab_size = tokenizer.vocab_size

print(f"Backend: {backend} | vocab_size: {vocab_size}")

BPE training: 100%|██████████| 744/744 [00:32<00:00, 22.94it/s]

Vocab size: 1000 | Merges: 744
Tokenizer saved → bpe_vocab.pkl
Backend: custom | vocab_size: 1000


In [3]:
# Load và encode toàn bộ dataset vào memory
with open(DATA_FILE, 'r', encoding='utf-8') as f:
    text = f.read()

text = text[:50000]   # ~50KB để train nhanh
print(f"Dataset: {len(text):,} ký tự")

data = torch.tensor(encode(text), dtype=torch.long)
print(f"Tokens : {len(data):,}  (trung bình {len(text)/len(data):.2f} ký tự/token)")

n          = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]
print(f"Train: {len(train_data):,} | Val: {len(val_data):,}")

def get_batch(split):
    d  = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x  = torch.stack([d[i:i+block_size]     for i in ix])
    y  = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

Dataset: 50,000 ký tự
Tokens : 18,085  (trung bình 2.76 ký tự/token)
Train: 16,276 | Val: 1,809


In [4]:
class CausalSelfAttention(nn.Module):
    """Multi-head causal self-attention với fused QKV projection"""

    def __init__(self, n_embd, n_head):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head  = n_head
        self.head_dim = n_embd // n_head
        self.c_attn  = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.c_proj  = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            'mask',
            torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
        )

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        out = F.scaled_dot_product_attention(
            q, k, v, is_causal=True,
            dropout_p=dropout if self.training else 0.0
        )
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.dropout(self.c_proj(out))


class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.ln1  = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head)
        self.ln2  = nn.LayerNorm(n_embd)
        self.ffwd = FeedForward(n_embd)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f   = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

        # Weight tying
        self.token_embedding_table.weight = self.lm_head.weight
        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        B, T    = index.shape
        tok_emb = self.token_embedding_table(index)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x       = self.blocks(tok_emb + pos_emb)
        logits  = self.lm_head(self.ln_f(x))
        loss    = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

    def generate(self, index, max_new_tokens, temperature=0.8, top_k=40):
        for _ in range(max_new_tokens):
            idx_cond = index[:, -block_size:]
            logits, _ = self.forward(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            index_next = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
            index = torch.cat((index, index_next), dim=1)
        return index


model = GPTLanguageModel(vocab_size).to(device)
print(f"Tham số: {sum(p.numel() for p in model.parameters()):,}")

Tham số: 737,920


In [5]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

optimizer = torch.optim.AdamW(
    model.parameters(), lr=learning_rate,
    betas=(0.9, 0.95), weight_decay=0.1
)

def get_lr(it):
    warmup_iters = 50
    min_lr = learning_rate / 10
    if it < warmup_iters:
        return learning_rate * (it + 1) / warmup_iters
    decay = (it - warmup_iters) / (max_iters - warmup_iters)
    return min_lr + 0.5 * (1.0 + math.cos(math.pi * decay)) * (learning_rate - min_lr)

for iter in range(max_iters):
    for pg in optimizer.param_groups:
        pg['lr'] = get_lr(iter)
    if iter % 50 == 0:
        losses = estimate_loss()
        print(f"step {iter:4d} | train: {losses['train']:.4f} | val: {losses['val']:.4f}")
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

print(f"\nFinal loss: {loss.item():.4f}")
with open('model-01.pkl', 'wb') as f:
    pickle.dump(model, f)
print('Model saved → model-01.pkl')

step    0 | train: 6.9323 | val: 6.9275
step   50 | train: 6.3213 | val: 6.3878
step  100 | train: 5.7740 | val: 6.0121
step  150 | train: 5.2673 | val: 5.6408
step  200 | train: 4.9217 | val: 5.3718
step  250 | train: 4.6426 | val: 5.2253
step  300 | train: 4.5094 | val: 5.1219
step  350 | train: 4.3931 | val: 5.0676
step  400 | train: 4.3286 | val: 5.0330
step  450 | train: 4.2838 | val: 5.0181

Final loss: 4.3156
Model saved → model-01.pkl


In [6]:
prompt = 'Hello! Can you see me?'
context = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
generated = decode(
    model.generate(context, max_new_tokens=100, temperature=0.8, top_k=40)[0].tolist()
)
print(generated)

Hello! Can you see me?  DTEHAAMding ste'tat that he could were iting to sidaint he he could you will glass in wealked duallos homentued the Dorothy
breaket and many cide.

Stones?"

Perror the horse saw the stes?"

"I am sans in floor but Dorothy's falling sasky 
